In [101]:
import seaborn as sns
import pandas as pd
import numpy as np

BÀI 1:

In [102]:
try:
    df = sns.load_dataset("titanic")
    print("Đã tải từ seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.columns = [c.lower() for c in df.columns]
    print("Đã tải từ URL.")
df.head()

Đã tải từ seaborn.


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [103]:
# Xử lý dữ liệu
leaky = ['class', 'who', 'adult_male', 'deck', 'embark_town','alive', 'alone']  
df = df.drop(columns=leaky)
print("Các cột còn lại:", list(df.columns))

Các cột còn lại: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']


In [104]:
# Chia train/val/test
from sklearn.model_selection import train_test_split


df_copy = df.copy()
X = df_copy.drop(columns = "survived")
y = df_copy["survived"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size= 0.15, train_size= 0.85, stratify=y)

print("Train/Test:", X_train.shape, X_test.shape)
# in tỷ lệ survived từng tập
def cnt_rate(s):
    cnt_0 = 0
    cnt_1 = 1
    for x in s:
        if x:
            cnt_1 +=1
        else:
            cnt_0 +=1
    return cnt_1/cnt_0
print("tỷ lệ survived tập train: ", cnt_rate(y_train))
print("tỷ lệ survived tập test: ", cnt_rate(y_test))

Train/Test: (757, 7) (134, 7)
tỷ lệ survived tập train:  0.6266094420600858
tỷ lệ survived tập test:  0.6265060240963856


In [105]:
# transform 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler


num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

pipe_so  = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  RobustScaler()),
])
pipe_cat = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder()),
])

preprocess = ColumnTransformer([
    ("num", pipe_so,  num_cols),
    ("cat", pipe_cat, cat_cols),
    ("ord", "passthrough", ord_cols),
])

X_train_t = preprocess.fit_transform(X_train)          # fit CHỈ trên train
X_test_t = preprocess.transform(X_test)

In [106]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
# Training mô hình Logistic Regression
Logistic_model = LogisticRegression()
Logistic_model.fit(X = X_train_t, y = y_train)
y_pred = Logistic_model.predict(X_test_t)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
confusion_matrix = confusion_matrix(y_test, y_pred)

print("accuracy là: ", accuracy)
print("precision là: ", precision)
print("recall là: ", recall)
print("f1 là: ", f1)
print("confusion_matrix là:\n", confusion_matrix)



accuracy là:  0.8059701492537313
precision là:  0.7272727272727273
recall là:  0.7843137254901961
f1 là:  0.7547169811320755
confusion_matrix là:
 [[68 15]
 [11 40]]


Vì đây là một dataset nhỏ với 891 sample và 7 features, ta có thể sử dụng KNN.

BÀI 2:

In [107]:
# Lấy dataset
from pathlib import Path

TRAIN_PATH = Path("Homework_b7/Dry_Bean_Dataset") / "dry_bean_train.csv"
TEST_PATH = Path("Homework_b7/Dry_Bean_Dataset") / "dry_bean_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df.head()

,area,perimeter,majoraxislength,minoraxislength,aspectration,eccentricity,convexarea,equivdiameter,extent,solidity,roundness,compactness,shapefactor1,shapefactor2,shapefactor3,shapefactor4,class
0,69471,1069.638,399.100245,225.005782,1.773733,0.825923,71088,297.410868,0.707386,0.977254,0.763027,0.745203,0.005745,0.001093,0.555328,0.985004,CALI
1,82877,1162.581,391.817013,270.836144,1.446694,0.722634,84171,324.841921,0.825986,0.984627,0.770544,0.829065,0.004728,0.001378,0.687349,0.994384,BARBUNYA
2,65042,1023.506,419.202858,198.962774,2.106941,0.880190,65748,287.774298,0.783403,0.989262,0.780231,0.686480,0.006445,0.000883,0.471255,0.992906,HOROZ
3,41315,758.920,287.438268,183.447580,1.566869,0.769858,41704,229.355383,0.791930,0.990672,0.901417,0.797929,0.006957,0.001740,0.636691,0.997611,SIRA
4,91088,1168.645,459.300729,253.950486,1.808623,0.833243,91799,340.553731,0.789051,0.992255,0.838119,0.741461,0.005042,0.000940,0.549765,0.994318,CALI


In [108]:
# Chia dữ liệu
X_train = train_df.drop(columns = "class")
y_train = train_df["class"]

X_test = test_df.drop(columns = "class")
y_test = test_df["class"]

In [109]:
# # Ta cần transform các biến categorical trong cột class thành số
# from sklearn.preprocessing import LabelEncoder


# transform = LabelEncoder()
# y_train = transform.fit_transform(y_train)
# y_test = transform.transform(y_test)

In [110]:
# Logistic Regression
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train, y_train,)
y_pred = logistic_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("accuracy là: ", accuracy)

accuracy là:  0.8770764119601329


d:\Coding\Code application\Python3139\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [111]:
# KNN
from sklearn.neighbors import KNeighborsClassifier
KNN_model = KNeighborsClassifier(n_neighbors = 8)
KNN_model.fit(X_train, y_train)
y_pred = KNN_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("accuracy là: ", accuracy)

accuracy là:  0.7216685123661868
